[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S11/S11_03_deteccion_segmentacion.ipynb)

# S11 · Práctica 3 — Detección de objetos y segmentación

**Tiempo estimado:** 1 h 45 min · **Sprint 11** (UD 4.6)

**Contexto TurisData.** Clasificar una foto dice *qué es*; pero al cliente le interesan preguntas como
*"¿cuántas sombrillas hay en la playa del resort?"* (detección) o *"¿qué parte de la imagen es piscina?"*
(segmentación). En esta práctica verás los conceptos, las métricas (IoU, NMS, mAP) con números pequeños y un
ejemplo completo **sin internet**, con imágenes aéreas sintéticas de una playa y con OpenCV.

**Al terminar sabrás:**
1. Distinguir clasificación, detección y segmentación (semántica frente a instancias).
2. Calcular la **IoU**, aplicar **NMS** y entender **precisión, recall, AP y mAP** con ejemplos numéricos.
3. Construir un detector clásico con OpenCV (color + contornos, y *template matching*) y evaluarlo.
4. Medir una segmentación (IoU/Dice de máscaras) y entrenar una mini red encoder-decoder.
5. Saber cuándo pasar a un detector moderno (YOLO) y qué cambia.

**Dos ramas.** Todo el notebook se ejecuta **offline**. Al final hay una celda **opcional de Colab con YOLO
(ultralytics)** marcada como *requiere internet* y **no verificada**.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os, sys, subprocess
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw

try:
    import cv2
except ImportError:      # en Colab y en el taller ya viene instalado; esta celda solo actúa si faltara
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"])
    import cv2

SEMILLA = 42
print("OpenCV", cv2.__version__)

## 1. Tres tareas con imágenes

| Tarea | Pregunta | Salida | Ejemplo TurisData |
|---|---|---|---|
| **Clasificación** | ¿Qué hay en la foto? | una etiqueta | "esta foto es de una piscina" |
| **Detección** | ¿Qué hay y **dónde**? | cajas (x1, y1, x2, y2) + clase + confianza | contar sombrillas en la playa |
| **Segmentación semántica** | ¿A qué clase pertenece **cada píxel**? | una máscara | superficie de piscina en la foto |
| **Segmentación de instancias** | Lo mismo, separando cada objeto | una máscara por objeto | cada tumbona por separado |

Los detectores modernos (YOLO, Faster R-CNN, DETR) son redes convolucionales que predicen muchas cajas candidatas de una vez;
después se limpian con **NMS** y se evalúan con **mAP**. Empezamos por esas piezas.

## 2. Cajas y la IoU (*Intersection over Union*)

**Idea.** Una caja se escribe `[x1, y1, x2, y2]` (esquina superior izquierda y esquina inferior derecha, en píxeles).
Para saber si una caja predicha "acierta" con la real, se mide cuánto se solapan:

`IoU = área de la intersección / área de la unión`

0 = no se tocan; 1 = idénticas. Por convención, una detección es correcta (verdadero positivo) si su IoU con una caja real es **≥ 0,5**.

**Ejemplo:** cajas `[0,0,10,10]` y `[5,5,15,15]`: intersección 5×5 = 25; unión 100 + 100 − 25 = 175; IoU = 25/175 ≈ 0,143.

**Ejercicio 1.** Escribe `iou(a, b)` que reciba dos cajas `[x1, y1, x2, y2]` y devuelva su IoU (un número entre 0 y 1).
Cuidado con el caso en que no se solapan (la intersección vale 0, no negativa).

In [ ]:
# TODO: ancho y alto de la intersección con max(0, ...); unión = área_a + área_b - intersección
...

assert abs(iou([0, 0, 10, 10], [5, 5, 15, 15]) - 25 / 175) < 1e-9, "IoU esperada 25/175 ≈ 0.143"
assert iou([0, 0, 10, 10], [0, 0, 10, 10]) == 1.0, "Cajas idénticas: IoU = 1"
assert iou([0, 0, 10, 10], [20, 20, 30, 30]) == 0.0, "Cajas separadas: IoU = 0 (¿has acotado con max(0, ...)?)"
assert abs(iou([0, 0, 10, 10], [0, 0, 10, 5]) - 0.5) < 1e-9, "Una caja dentro de otra: IoU = área pequeña / área grande"

## 3. NMS (*Non-Maximum Suppression*): quedarse con una caja por objeto

**Idea.** Un detector suele proponer **varias cajas casi iguales** para el mismo objeto. NMS las limpia:
1. Ordena las cajas por confianza (de mayor a menor).
2. Se queda con la primera.
3. **Descarta** las demás que se solapan con ella más de un umbral (por ejemplo IoU > 0,5).
4. Repite con las que quedan.

**Ejemplo:** cajas A (confianza 0,9), B (0,8, casi igual a A) y C (0,7, lejos). NMS conserva A y C: B era un duplicado de A.

**Ejercicio 2.** Escribe `nms(cajas, puntuaciones, umbral_iou=0.5)` que devuelva la **lista de índices** de las cajas que se
conservan, en orden de confianza decreciente. Usa tu función `iou`.

In [ ]:
# TODO: Ordena índices por puntuación; mientras queden, guarda el primero y filtra los que tienen iou > umbral con él
...

_cajas = [[10, 10, 50, 50], [12, 12, 52, 52], [100, 100, 140, 140], [11, 9, 49, 51]]
_punt = [0.9, 0.8, 0.7, 0.6]
assert nms(_cajas, _punt) == [0, 2], "Debe quedar la caja 0 (mejor) y la 2 (lejana); la 1 y la 3 son duplicados de la 0"
assert nms(_cajas, _punt, umbral_iou=0.99) == [0, 1, 2, 3], "Con un umbral muy alto no se descarta nada"
assert nms([[0, 0, 5, 5]], [0.3]) == [0], "Con una sola caja se conserva"

## 4. Métricas: precisión, recall, AP y mAP con números

Imagina que en una foto hay **4 sombrillas reales** y el detector entrega 5 cajas con su confianza. Tras ordenarlas por
confianza y compararlas con la realidad (IoU ≥ 0,5), quedan así (✔ = verdadero positivo, ✘ = falso positivo):

| Puesto | Confianza | ¿Correcta? | TP acumulados | Precisión acumulada | Recall acumulado |
|---|---|---|---|---|---|
| 1 | 0,95 | ✔ | 1 | 1/1 = 1,00 | 1/4 = 0,25 |
| 2 | 0,90 | ✔ | 2 | 2/2 = 1,00 | 2/4 = 0,50 |
| 3 | 0,80 | ✘ | 2 | 2/3 = 0,67 | 2/4 = 0,50 |
| 4 | 0,70 | ✔ | 3 | 3/4 = 0,75 | 3/4 = 0,75 |
| 5 | 0,60 | ✘ | 3 | 3/5 = 0,60 | 3/4 = 0,75 |

- **Precisión:** de lo que el detector dice, ¿cuánto es cierto? **Recall:** de lo que hay, ¿cuánto encuentra?
- Al bajar el umbral de confianza suben los aciertos (recall) pero también las falsas alarmas (precisión baja): la curva
  **precisión–recall** describe ese compromiso.
- **AP (*average precision*)** = área bajo esa curva (con la precisión "suavizada": en cada punto se toma la mayor precisión
  que se consigue con ese recall o más). Aquí: 0,25×1,00 + 0,25×1,00 + 0,25×0,75 = **0,6875**. (La 5.ª detección no añade recall.)
- **mAP** = media del AP de **todas las clases** (y, en el estándar COCO, también promediada sobre varios umbrales de IoU de 0,5 a 0,95:
  *mAP@[.5:.95]*). Es la cifra con la que se comparan detectores.

**Ejercicio 3.** Escribe `average_precision(es_tp, n_reales)`: recibe una lista de 0/1 con las detecciones **ya ordenadas por
confianza decreciente** (1 = verdadero positivo) y el número de objetos reales `n_reales`, y devuelve el AP.
Pasos: acumula TP y FP; calcula `precision = TP/(TP+FP)` y `recall = TP/n_reales` en cada puesto; haz la precisión
monótona (de derecha a izquierda, cada valor pasa a ser el máximo de los que tiene a su derecha); suma
`(recall_i − recall_{i−1}) × precision_i`.

In [ ]:
# TODO: np.cumsum para TP y FP; la precisión se "suaviza" de derecha a izquierda; suma ponderada por el aumento de recall
...

assert abs(average_precision([1, 1, 0, 1, 0], 4) - 0.6875) < 1e-9, "Con la tabla del enunciado el AP debe ser 0.6875"
assert abs(average_precision([1, 1, 1, 1], 4) - 1.0) < 1e-9, "Detector perfecto: AP = 1"
assert abs(average_precision([1, 1], 4) - 0.5) < 1e-9, "Solo encuentra la mitad de los objetos: AP = 0.5"
assert average_precision([0, 0, 0], 3) == 0.0, "Sin ningún acierto el AP es 0"

## 5. Un detector clásico con OpenCV sobre imágenes sintéticas

**Escena.** Vista aérea esquemática de la playa de un resort (128×128 píxeles): arena, franja de mar, **sombrillas**
(círculos rojos, naranjas, amarillos o de un amarillo pastel poco saturado, a veces solapados) y **distractores** (palmeras verdes, tumbonas grises). El generador también entrega
las **cajas reales** y la **máscara** de sombrillas. Son dibujos, no fotos.

In [ ]:
def _col(rng, base, var=12):
    return tuple(int(np.clip(c + rng.integers(-var, var + 1), 0, 255)) for c in base)

def escena(rng, tam=128, n_max=8):
    """Devuelve (imagen RGB uint8, cajas reales [x1,y1,x2,y2], máscara booleana de sombrillas)."""
    img = Image.new("RGB", (tam, tam), _col(rng, (230, 210, 160)))
    d = ImageDraw.Draw(img)
    mar = int(tam * rng.uniform(0.15, 0.28))
    d.rectangle([0, 0, tam, mar], fill=_col(rng, (60, 140, 200), 15))
    masc = Image.new("L", (tam, tam), 0)
    dm = ImageDraw.Draw(masc)
    ocupados, cajas = [], []
    n = int(rng.integers(3, n_max + 1))
    for _ in range(300):
        if len(cajas) >= n + int(rng.integers(2, 5)) and len(cajas) >= n:
            break
        r = int(rng.integers(6, 10))
        cx, cy = int(rng.integers(r + 2, tam - r - 2)), int(rng.integers(mar + r + 3, tam - r - 2))
        if any(np.hypot(cx - x, cy - y) < r + rr - 4 for x, y, rr, _ in ocupados):   # se permite algo de solape
            continue
        if len([o for o in ocupados if o[3] == "s"]) < n:                      # sombrilla
            color = [(220, 40, 40), (240, 140, 30), (240, 210, 40), (245, 225, 140)][int(rng.integers(0, 4))]   # la 4.ª es pastel (poco saturada)
            d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=_col(rng, color, 10))
            d.ellipse([cx - 1, cy - 1, cx + 1, cy + 1], fill=(90, 60, 40))
            dm.ellipse([cx - r, cy - r, cx + r, cy + r], fill=255)
            cajas.append([cx - r, cy - r, cx + r, cy + r]); ocupados.append((cx, cy, r, "s"))
        elif rng.random() < 0.5:                                                # palmera (distractor verde)
            d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=_col(rng, (30, 140, 60), 15)); ocupados.append((cx, cy, r, "p"))
        else:                                                                   # tumbona (distractor gris)
            d.rectangle([cx - r, cy - 3, cx + r, cy + 3], fill=_col(rng, (150, 150, 155), 15)); ocupados.append((cx, cy, r, "t"))
    a = np.asarray(img, dtype="float32") + rng.normal(0, 6, (tam, tam, 3))
    return np.clip(a, 0, 255).astype("uint8"), np.array(cajas), np.asarray(masc) > 0

def dibujar_cajas(eje, cajas, color, etiquetas=None):
    for k, (x1, y1, x2, y2) in enumerate(cajas):
        eje.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, lw=1.6))
        if etiquetas is not None:
            eje.text(x1, y1 - 1, etiquetas[k], color=color, fontsize=7, va="bottom")

rng = np.random.default_rng(SEMILLA)
img0, cajas0, masc0 = escena(rng)
fig, ejes = plt.subplots(1, 2, figsize=(9, 4.4))
ejes[0].imshow(img0); ejes[0].set_title("Escena sintética"); ejes[0].axis("off")
ejes[1].imshow(img0); dibujar_cajas(ejes[1], cajas0, "lime"); ejes[1].set_title(f"Cajas reales ({len(cajas0)} sombrillas)"); ejes[1].axis("off")
plt.tight_layout(); plt.show()

### Detector 1: *template matching* (comparar con una plantilla) + NMS
Se desliza una **plantilla** (un círculo saturado) por la imagen y se mide el parecido en cada posición
(`cv2.matchTemplate`). Cada sombrilla aparece con **muchas posiciones casi iguales** que superan el umbral: justo el caso
para el que sirve el NMS. Trabajamos sobre el canal de **saturación** (los colores vivos destacan sobre la arena y no dependen del tono).

In [ ]:
def saturacion(img_rgb):
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)[:, :, 1]

plantilla_img = Image.new("RGB", (24, 24), (230, 210, 160))
ImageDraw.Draw(plantilla_img).ellipse([4, 4, 20, 20], fill=(230, 100, 30))
plantilla = saturacion(np.asarray(plantilla_img))
LADO = 16    # caja que asignamos a cada acierto (diámetro aproximado de una sombrilla)

def detectar_plantilla(img_rgb, umbral=0.6):
    res = cv2.matchTemplate(saturacion(img_rgb), plantilla, cv2.TM_CCOEFF_NORMED)
    ys, xs = np.where(res >= umbral)
    cajas = [[x + 4, y + 4, x + 4 + LADO, y + 4 + LADO] for x, y in zip(xs, ys)]      # el círculo está a 4 px del borde de la plantilla
    return cajas, [float(res[y, x]) for x, y in zip(xs, ys)]

cajas_brutas, punt_brutas = detectar_plantilla(img0)
quedan = nms(cajas_brutas, punt_brutas, umbral_iou=0.3)
print(f"Detecciones brutas: {len(cajas_brutas)} → tras NMS: {len(quedan)} (sombrillas reales: {len(cajas0)})")

fig, ejes = plt.subplots(1, 2, figsize=(9, 4.4))
for eje, cajas, titulo in [(ejes[0], cajas_brutas, f"Antes de NMS ({len(cajas_brutas)} cajas)"),
                           (ejes[1], [cajas_brutas[i] for i in quedan], f"Después de NMS ({len(quedan)} cajas)")]:
    eje.imshow(img0); dibujar_cajas(eje, cajas, "red"); dibujar_cajas(eje, cajas0, "lime"); eje.set_title(titulo); eje.axis("off")
plt.suptitle("Rojo = detecciones · verde = cajas reales"); plt.tight_layout(); plt.show()

### Detector 2: color + contornos
Otra idea clásica: (1) pasar a HSV, (2) quedarse con los píxeles de color vivo y tono rojo–amarillo (**máscara**),
(3) limpiar el ruido con una **apertura morfológica**, (4) encontrar los **contornos** (`cv2.findContours`) y (5) quedarse con
la caja que rodea a cada contorno de tamaño razonable. La confianza será lo "circular" que es el contorno.

In [ ]:
def detectar_color(img_rgb, sat_min=120, area_min=40):
    """Devuelve (cajas, confianzas, máscara) de zonas de color vivo rojo–naranja–amarillo."""
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    h, s, v = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
    mascara = (((h <= 35) | (h >= 170)) & (s >= sat_min) & (v >= 120)).astype("uint8") * 255
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cajas, conf = [], []
    for c in contornos:
        area = cv2.contourArea(c)
        if area < area_min:
            continue
        x, y, w, h_ = cv2.boundingRect(c)
        cajas.append([x, y, x + w, y + h_])
        conf.append(float(min(1.0, area / (np.pi / 4 * w * h_))))          # 1 = perfectamente circular/relleno
    return cajas, conf, mascara > 0

cajas_c, conf_c, masc_c = detectar_color(img0)
fig, ejes = plt.subplots(1, 3, figsize=(13, 4.2))
ejes[0].imshow(img0); ejes[0].set_title("Imagen"); ejes[0].axis("off")
ejes[1].imshow(masc_c, cmap="gray"); ejes[1].set_title("Máscara de color vivo (tras apertura)"); ejes[1].axis("off")
ejes[2].imshow(img0); dibujar_cajas(ejes[2], cajas_c, "red", [f"{c:.2f}" for c in conf_c]); dibujar_cajas(ejes[2], cajas0, "lime")
ejes[2].set_title("Cajas del detector (rojo) y reales (verde)"); ejes[2].axis("off")
plt.tight_layout(); plt.show()

### Evaluar el detector: emparejar detecciones con la realidad
Para calcular precisión, recall y AP hay que decidir qué detecciones son verdaderos positivos: se recorren **por confianza
decreciente** y cada una se empareja con la caja real **libre** de mayor IoU, si esta supera el umbral. Una caja real solo puede
emparejarse una vez (las duplicadas cuentan como falsos positivos).

In [ ]:
def emparejar(cajas_det, conf, cajas_reales, umbral_iou=0.5):
    """Devuelve la lista de 0/1 (TP/FP) de las detecciones ordenadas por confianza decreciente."""
    orden = np.argsort(conf)[::-1]
    usadas, es_tp = set(), []
    for i in orden:
        mejor, mejor_iou = None, umbral_iou
        for j, real in enumerate(cajas_reales):
            u = iou(cajas_det[i], real)
            if j not in usadas and u >= mejor_iou:
                mejor, mejor_iou = j, u
        if mejor is not None:
            usadas.add(mejor); es_tp.append(1)
        else:
            es_tp.append(0)
    return es_tp

def evaluar_detector(detector, n_escenas=30, semilla=100, umbral_iou=0.5):
    """AP, precisión y recall globales sobre n_escenas nuevas. `detector(img)` devuelve (cajas, confianzas)."""
    rng = np.random.default_rng(semilla)
    todos_conf, todos_tp, n_reales, n_detectadas = [], [], 0, 0
    for _ in range(n_escenas):
        img, reales, _ = escena(rng)
        cajas, conf = detector(img)
        n_reales += len(reales); n_detectadas += len(cajas)
        tp = emparejar(cajas, conf, reales, umbral_iou) if len(cajas) else []
        orden = np.argsort(conf)[::-1] if len(cajas) else []
        todos_conf += [conf[i] for i in orden]; todos_tp += tp
    orden = np.argsort(todos_conf)[::-1]
    tp_global = [todos_tp[i] for i in orden]
    return {"AP": average_precision(tp_global, n_reales),
            "precisión": sum(tp_global) / max(1, n_detectadas), "recall": sum(tp_global) / n_reales,
            "reales": n_reales, "detectadas": n_detectadas}

def detector_color(img):
    cajas, conf, _ = detectar_color(img)
    return cajas, conf

resultado = evaluar_detector(detector_color)
print({k: round(v, 3) if isinstance(v, float) else v for k, v in resultado.items()})

**Ejercicio 4.** Calcula el **mAP@[.5:.9]** simplificado del detector de color: la media del AP con umbrales de IoU
`0.5, 0.6, 0.7, 0.8, 0.9` (usa `evaluar_detector(detector_color, umbral_iou=u)["AP"]`). Guarda la media en `map_iou`
y, en `ap_por_umbral`, la lista de los 5 AP. Como solo hay una clase (sombrilla), el mAP coincide con el AP.

In [ ]:
# TODO: Un bucle sobre los umbrales llamando a evaluar_detector; np.mean de la lista
...

assert len(ap_por_umbral) == 5, "Debe haber un AP por cada umbral (5 en total)"
assert all(ap_por_umbral[i] >= ap_por_umbral[i + 1] - 1e-9 for i in range(4)), "El AP no debería subir al exigir más IoU. Revisa el orden de los umbrales"
assert 0 <= map_iou <= 1, "El mAP es un número entre 0 y 1"

plt.plot(umbrales, ap_por_umbral, "o-"); plt.xlabel("umbral de IoU exigido"); plt.ylabel("AP")
plt.title(f"Detector de color: AP según el umbral de IoU (media = {map_iou:.2f})"); plt.ylim(0, 1.02); plt.show()

## 6. Segmentación

**Idea.** Ahora no buscamos cajas sino una **máscara**: para cada píxel, ¿es sombrilla o no? La métrica natural es
la IoU de máscaras (o el **coeficiente Dice**, muy usado en medicina):

- `IoU = |A ∩ B| / |A ∪ B|` · `Dice = 2·|A ∩ B| / (|A| + |B|)`

La máscara que produce nuestro detector de color ya es una segmentación: la comparamos con la máscara real.

**Ejercicio 5.** Escribe `iou_mascara(pred, real)` y `dice(pred, real)` para máscaras booleanas del mismo tamaño
(`True` = sombrilla). Devuelve 1.0 si ambas están vacías.

In [ ]:
# TODO: Usa np.logical_and / np.logical_or y suma los True (.sum())
...

_p = np.array([[1, 1, 0], [0, 0, 0]], dtype=bool); _r = np.array([[1, 0, 0], [1, 0, 0]], dtype=bool)
assert abs(iou_mascara(_p, _r) - 1 / 3) < 1e-9, "Intersección 1, unión 3 → 1/3"
assert abs(dice(_p, _r) - 0.5) < 1e-9, "2·1 / (2 + 2) = 0.5"
assert iou_mascara(np.zeros((2, 2), bool), np.zeros((2, 2), bool)) == 1.0, "Dos máscaras vacías coinciden: 1.0"

In [ ]:
rng = np.random.default_rng(200)
ious_umbral = []
for _ in range(20):
    im, _, real = escena(rng)
    _, _, pred = detectar_color(im)
    ious_umbral.append((iou_mascara(pred, real), dice(pred, real)))
print("Segmentación por umbral de color: IoU media =", round(np.mean([a for a, _ in ious_umbral]), 3),
      "· Dice medio =", round(np.mean([b for _, b in ious_umbral]), 3))

### Segmentación por agrupación de colores (*k-means*)
Sin entrenar ningún modelo: se agrupan los píxeles en *k* colores. Probamos con una foto de ejemplo que trae scikit-learn
(`load_sample_images`, incluida en la instalación: no necesita internet).

In [ ]:
from sklearn.datasets import load_sample_images
foto = load_sample_images().images[1]                                # flor (427×640×3)
peq = cv2.resize(foto, (160, 107)).reshape(-1, 3).astype("float32")
criterio = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
cv2.setRNGSeed(SEMILLA)
_, etiquetas, centros = cv2.kmeans(peq, 4, None, criterio, 3, cv2.KMEANS_PP_CENTERS)
segmentada = centros.astype("uint8")[etiquetas.ravel()].reshape(107, 160, 3)
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.6))
ejes[0].imshow(foto); ejes[0].set_title("Foto original"); ejes[0].axis("off")
ejes[1].imshow(segmentada); ejes[1].set_title("Segmentada con k-means (4 colores)"); ejes[1].axis("off")
plt.tight_layout(); plt.show()

### Una mini red de segmentación (encoder-decoder)
Las redes modernas de segmentación (U-Net, DeepLab) tienen dos mitades: un **codificador** que reduce la imagen extrayendo
rasgos (convolución + pooling) y un **decodificador** que la vuelve a agrandar (`UpSampling`) hasta dar **una probabilidad por
píxel**. Aquí una versión diminuta entrenada en 300 escenas sintéticas para "pintar" las sombrillas.

In [ ]:
import tensorflow as tf
from tensorflow import keras
keras.utils.set_random_seed(SEMILLA)

rng = np.random.default_rng(300)
datos = [escena(rng) for _ in range(360)]
X_seg = np.stack([d[0] for d in datos]).astype("float32") / 255
Y_seg = np.stack([d[2] for d in datos]).astype("float32")[..., None]
X_s_ent, Y_s_ent, X_s_pru, Y_s_pru = X_seg[:300], Y_seg[:300], X_seg[300:], Y_seg[300:]

def construir_segmentador():
    e = keras.layers.Input(shape=(128, 128, 3))
    x = keras.layers.Conv2D(8, 3, padding="same", activation="relu")(e)
    x = keras.layers.MaxPooling2D()(x)                                         # 64×64
    x = keras.layers.Conv2D(16, 3, padding="same", activation="relu")(x)
    x = keras.layers.UpSampling2D()(x)                                         # 128×128
    x = keras.layers.Conv2D(8, 3, padding="same", activation="relu")(x)
    s = keras.layers.Conv2D(1, 1, activation="sigmoid")(x)                      # probabilidad por píxel
    m = keras.Model(e, s)
    m.compile("adam", "binary_crossentropy")
    return m

seg = construir_segmentador()
h_seg = seg.fit(X_s_ent, Y_s_ent, epochs=12, batch_size=16, validation_split=0.1, verbose=0)
pred_seg = seg.predict(X_s_pru, verbose=0)[..., 0] > 0.5
iou_red = np.mean([iou_mascara(p, r[..., 0] > 0) for p, r in zip(pred_seg, Y_s_pru)])
iou_umb = np.mean([iou_mascara(detectar_color((x * 255).astype("uint8"))[2], r[..., 0] > 0) for x, r in zip(X_s_pru, Y_s_pru)])
print(f"IoU de máscaras en 60 escenas nuevas → mini red: {iou_red:.3f} · umbral de color: {iou_umb:.3f}")

fig, ejes = plt.subplots(1, 3, figsize=(11, 3.8))
ejes[0].imshow(X_s_pru[0]); ejes[0].set_title("Imagen"); ejes[1].imshow(Y_s_pru[0, ..., 0], cmap="gray"); ejes[1].set_title("Máscara real")
ejes[2].imshow(pred_seg[0], cmap="gray"); ejes[2].set_title("Predicción de la red")
for e in ejes: e.axis("off")
plt.tight_layout(); plt.show()

**Qué observar.** En un problema tan limpio (color inconfundible) un umbral de color puede igualar o superar a una red pequeña:
**un método clásico bien pensado es un buen punto de partida**. Las redes ganan cuando los objetos varían mucho de aspecto,
hay sombras, oclusiones o fondos complejos, donde una regla de color ya no basta.

## 7. Colab (opcional): un detector moderno con YOLO

> **AVISO: esta celda requiere internet y NO se ha verificado en el taller.** Instala la librería `ultralytics`, descarga
> los pesos preentrenados `yolov8n.pt` (≈ 6 MB, entrenados en COCO: 80 clases como persona, coche, silla, paraguas) y una
> imagen de ejemplo. Pon `USAR_INTERNET = True` en Colab. Fíjate en que YOLO devuelve **cajas + clase + confianza** y que
> ya aplica NMS internamente (parámetro `iou`).

In [ ]:
USAR_INTERNET = False        # ← True solo con conexión (Colab)

if USAR_INTERNET:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    from ultralytics import YOLO
    modelo_yolo = YOLO("yolov8n.pt")                                                     # descarga los pesos
    resultados = modelo_yolo("https://ultralytics.com/images/bus.jpg", conf=0.25, iou=0.5)    # descarga la imagen
    r = resultados[0]
    plt.imshow(r.plot()[:, :, ::-1]); plt.axis("off"); plt.title("YOLOv8n (COCO)"); plt.show()
    for caja, cls, c in zip(r.boxes.xyxy.tolist(), r.boxes.cls.tolist(), r.boxes.conf.tolist()):
        print(r.names[int(cls)], round(c, 2), [round(v) for v in caja])
    # Para tu dataset propio: modelo_yolo.train(data="datos.yaml", epochs=30, imgsz=640)  (requiere etiquetas en formato YOLO)
else:
    print("Rama de Colab desactivada (USAR_INTERNET = False). El resto del notebook no la necesita.")

**Qué cambia con un detector moderno:** aprende el detector completo a partir de miles de imágenes etiquetadas con cajas
(etiquetar cajas cuesta más que etiquetar clases), predice cientos de cajas por imagen en una sola pasada y se evalúa con
**mAP@[.5:.95]**. Lo que has hecho a mano (IoU, NMS, AP) es exactamente lo que ocurre dentro y alrededor de YOLO.

## 8. Mini-reto integrador: contar sombrillas y afinar el detector

El cliente quiere un **contador de sombrillas** por foto de la playa. Con 30 escenas nuevas (`semilla=500`):
1. Calcula en `error_conteo` el error absoluto medio entre el número de cajas del `detector_color` y el número real.
2. Prueba `sat_min` ∈ {60, 90, 120, 150, 180} y guarda en `mejor_sat` el valor que da mayor **F1** de detección
   (F1 = 2·P·R/(P+R), con IoU ≥ 0,5, usando `evaluar_detector` con un detector que fije ese `sat_min`) y en `f1_mejor` su valor.
Con sat_min bajo la arena "cuela" como sombrilla; con sat_min alto se pierden sombrillas amarillas.

In [ ]:
# TODO: Cuenta len(cajas) frente a len(reales) por escena; para el F1 usa functools.partial o una lambda con sat_min
...

assert error_conteo >= 0, "El error de conteo no puede ser negativo"
assert mejor_sat in (60, 90, 120, 150, 180), "mejor_sat debe ser uno de los valores probados"
assert f1_mejor > 0.5, "Con el mejor umbral el F1 debería superar 0.5. Revisa cómo calculas precisión y recall"
print(f"Error medio de conteo: {error_conteo:.2f} sombrillas por foto · mejor sat_min = {mejor_sat} (F1 = {f1_mejor:.2f})")
print({s: round(v, 2) for s, v in f1_por_sat.items()})

**Tu análisis (edita esta celda):** ¿en qué fotos falla el contador (sombrillas pegadas, colores parecidos a la arena)?
¿Qué pasaría si las sombrillas fueran blancas? ¿Qué harías para que funcione con fotos reales?

## 9. Preguntas de reflexión
1. Un detector con AP = 0,9 a IoU 0,5 pero 0,4 a IoU 0,9: ¿qué te dice sobre la calidad de sus cajas?
2. ¿Por qué sin NMS un detector cuenta varias veces el mismo objeto? ¿Qué pasaría con un umbral de NMS demasiado bajo si hay objetos muy juntos?
3. ¿Cuándo elegirías segmentación en vez de detección para TurisData (superficie de piscina, ocupación de la playa…)?
4. Etiquetar cajas es más caro que etiquetar clases. ¿Cómo influiría en la decisión de usar YOLO?

## 10. Resumen: qué has aprendido
- Clasificar = qué; detectar = qué y dónde (cajas); segmentar = qué es cada píxel.
- **IoU** mide el solape; **NMS** elimina duplicados; **AP/mAP** resumen el compromiso precisión–recall (y mAP@[.5:.95] exige cajas precisas).
- Un detector clásico (color + contornos, *template matching*) sirve de referencia y se evalúa con las mismas métricas que uno moderno.
- La segmentación se mide con IoU/Dice de máscaras; un encoder-decoder diminuto ya aprende a pintar objetos sencillos.
- YOLO automatiza todo el proceso con redes profundas, a cambio de necesitar muchas imágenes etiquetadas con cajas.